# Run the factorial experiment

36 runs: 4 architectures x 3 split arms x 3 seeds.

Resumable. Results append to `results_nlst.csv` on **Drive**, so a Colab
disconnect costs at most the run in progress. Re-run cell 3 to continue.

Set the runtime to **GPU** first: Runtime -> Change runtime type -> T4 GPU.

In [ ]:
import tensorflow as tf
print("GPU:", tf.config.list_physical_devices('GPU'))

GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!git clone -q https://github.com/elhamum/lungcnet-leakage.git
%cd /content/lungcnet-leakage
!pip install -q pyyaml

import tensorflow as tf, os
print("GPU:", tf.config.list_physical_devices("GPU") or "NONE -- enable the T4!")

os.makedirs("/content/drive/MyDrive/leakage_bench", exist_ok=True)
RESULTS = "/content/drive/MyDrive/leakage_bench/results_nlst.csv"
print("results ->", RESULTS)

Mounted at /content/drive
/content/lungcnet-leakage
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
results -> /content/drive/MyDrive/leakage_bench/results_nlst.csv


## 2. Freeze the splits

Regenerates the same nine arm x seed columns from the committed manifest.
Deterministic, so it reproduces the splits already in the repo.

In [ ]:
!python src/make_splits.py --config configs/nlst.yaml

wrote data/splits_nlst.csv  (5,810 images x 9 split columns)

partition sizes:
    arm partition  n_images  pct_of_pool  n_benign  n_normal  n_suspicious  seed
   exam      test      1160    19.965577       460       450           250     0
   exam     train      4070    70.051635      1620      1580           870     0
   exam       val       580     9.982788       230       230           120     0
patient      test      1170    20.137694       470       460           240     0
patient     train      4060    69.879518      1610      1570           880     0
patient       val       580     9.982788       230       230           120     0
  slice      test      1162    20.000000       462       452           248     0
  slice     train      4067    70.000000      1617      1582           868     0
  slice       val       581    10.000000       231       226           124     0
   exam      test      1160    19.965577       460       450           250     1
   exam     train      4070   

## 3. Run the queue

Arms sweep innermost, so an interrupted queue still leaves complete arm
comparisons for the architectures it finished.

Add `--limit 6` to cap how many runs happen in one session.

In [ ]:
!PYTHONPATH=src python src/runner.py --config configs/nlst.yaml --results "/content/drive/MyDrive/leakage_bench/results_nlst.csv" --limit 10

nlst: 36 runs in grid, 30 done, 6 remaining

[1/6] nlst | mobilenetv2 | slice | seed 1
2026-09-03 11:26:48.650340: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
I0000 00:00:1788434808.651746   14847 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
/content/lungcnet-leakage/src/models.py:31: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base = cfg["fn"](weights="imagenet", include_top=False,
Epoch 1/50
2026-09-03 11:27:05.970207: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution,

In [ ]:
import pandas as pd
r = pd.read_csv("/content/drive/MyDrive/leakage_bench/results_nlst.csv")
print(len(r), "runs")
print(r[["arch","arm","seed","test_macro_f1","train_seconds","finished_utc"]].tail(6).to_string(index=False))

23 runs
       arch     arm  seed  test_macro_f1  train_seconds              finished_utc
      vgg16 patient     2          34.02          264.0 2026-09-01T09:58:05+00:00
inceptionv3   slice     0          56.53          265.9 2026-09-01T10:02:31+00:00
inceptionv3    exam     0          41.54          171.9 2026-09-01T10:05:23+00:00
inceptionv3 patient     0          29.49         5209.2 2026-09-02T08:24:06+00:00
inceptionv3   slice     1          56.26          968.0 2026-09-03T06:58:33+00:00
inceptionv3    exam     1          45.06        13555.2 2026-09-03T10:44:28+00:00


In [ ]:
import csv, shutil

P = "/content/drive/MyDrive/leakage_bench/results_nlst.csv"
shutil.copy(P, P + ".backup")          # keep the original, just in case

OLD = ['dataset','arch','arm','seed','n_train','n_val','n_test','test_accuracy',
       'test_macro_f1','test_macro_auc','test_loss','val_accuracy','val_loss',
       'per_class_f1_json','confusion_json','epochs_run','best_epoch',
       'collapsed_to_single_class','train_seconds','manifest_sha1','split_sha1',
       'git_commit','started_utc','finished_utc','status','note']
NEW = OLD[:18] + ['classes_never_predicted','majority_baseline'] + OLD[18:]

with open(P, newline='') as fh:
    rows = list(csv.reader(fh))

body = [r for r in rows[1:] if r]
fixed = [r[:18] + ['',''] + r[18:] if len(r) == 26 else r for r in body]

bad = [r for r in fixed if len(r) != 28]
print(f"{len(fixed)} rows, {len(bad)} still malformed")

with open(P, 'w', newline='') as fh:
    w = csv.writer(fh)
    w.writerow(NEW)
    w.writerows(fixed)

import pandas as pd
r = pd.read_csv(P)
print(f"\n{len(r)} runs recorded")
print(r[["arch","arm","seed","status","test_macro_f1","train_seconds"]].to_string(index=False))

36 rows, 0 still malformed

36 runs recorded
       arch     arm  seed status  test_macro_f1  train_seconds
   lungcnet   slice     0     ok          61.85         1015.6
   lungcnet    exam     0     ok          35.83          116.7
   lungcnet patient     0     ok          28.17          124.3
   lungcnet   slice     1     ok          63.67          298.0
   lungcnet    exam     1     ok          36.47           83.5
   lungcnet patient     1     ok          31.47          571.4
   lungcnet   slice     2     ok          62.16          335.0
   lungcnet    exam     2     ok          34.17          122.0
   lungcnet patient     2     ok          31.33          113.4
      vgg16   slice     0     ok          72.51          664.5
      vgg16    exam     0     ok          36.96          263.4
      vgg16 patient     0     ok          30.12          275.0
      vgg16   slice     1     ok          77.20         1751.8
      vgg16    exam     1     ok          36.02          319.5
      vgg1

In [ ]:
import pandas as pd
r = pd.read_csv("/content/drive/MyDrive/leakage_bench/results_nlst.csv")
for _, x in r.iterrows():
    print(f"{x['arm']:8s} seed{x['seed']}  F1={x['test_macro_f1']:5.2f}  "
          f"epochs={x['epochs_run']:2d}  collapsed={x['collapsed_to_single_class']}")
    print("   per-class:", x["per_class_f1_json"])
    print("   confusion:", x["confusion_json"])

slice    seed0  F1=61.85  epochs=22  collapsed=False
   per-class: {"benign": 66.12, "normal": 67.09, "suspicious": 52.35}
   confusion: [[324, 107, 31], [117, 315, 20], [77, 65, 106]]
exam     seed0  F1=35.83  epochs= 7  collapsed=False
   per-class: {"benign": 55.94, "normal": 51.57, "suspicious": 0.0}
   confusion: [[311, 149, 0], [203, 247, 0], [138, 112, 0]]
patient  seed0  F1=28.17  epochs= 6  collapsed=False
   per-class: {"benign": 50.04, "normal": 34.46, "suspicious": 0.0}
   confusion: [[296, 174, 0], [302, 158, 0], [115, 125, 0]]
slice    seed1  F1=63.67  epochs=20  collapsed=False
   per-class: {"benign": 69.22, "normal": 69.44, "suspicious": 52.36}
   confusion: [[344, 66, 52], [108, 300, 44], [80, 46, 122]]
exam     seed1  F1=36.47  epochs= 6  collapsed=False
   per-class: {"benign": 58.06, "normal": 51.35, "suspicious": 0.0}
   confusion: [[342, 118, 0], [221, 229, 0], [155, 95, 0]]
patient  seed1  F1=31.47  epochs= 6  collapsed=False
   per-class: {"benign": 49.13, "nor

In [ ]:
import os
print("drive mounted:", os.path.exists("/content/drive/MyDrive"))
print(os.listdir("/content/drive/MyDrive/leakage_bench") if os.path.exists("/content/drive/MyDrive/leakage_bench") else "folder not there")

drive mounted: True
['nlst', 'results_nlst.csv']


## 4. Where things stand

In [ ]:
import pandas as pd
r = pd.read_csv(RESULTS)
print(f"{len(r)} runs recorded, {(r.status=='ok').sum()} ok\n")

ok = r[r.status == "ok"]
if len(ok):
    print("macro F1 by architecture and split arm:")
    print(ok.pivot_table(index="arch", columns="arm", values="test_macro_f1",
                         aggfunc=["mean","std","count"]).round(2).to_string())

    m = ok.groupby(["arch","arm"]).test_macro_f1.mean().unstack()
    if {"slice","patient"}.issubset(m.columns):
        print("\ninflation from leakage (macro F1 points):")
        cols = {"slice - patient": (m["slice"] - m["patient"]).round(2)}
        if "exam" in m.columns:
            cols["exam - patient"] = (m["exam"] - m["patient"]).round(2)
        print(pd.DataFrame(cols).to_string())

    if ok.collapsed_to_single_class.any():
        print("\ncollapsed runs (report these, do not silently average them):")
        print(ok[ok.collapsed_to_single_class]
              [["arch","arm","seed","val_accuracy"]].to_string(index=False))

36 runs recorded, 36 ok

macro F1 by architecture and split arm:
              mean                  std                count              
arm           exam patient  slice  exam patient  slice  exam patient slice
arch                                                                      
inceptionv3  44.57   33.29  56.56  2.81    3.49   0.32     3       3     3
lungcnet     35.49   30.32  62.56  1.19    1.87   0.97     3       3     3
mobilenetv2  41.04   35.47  54.09  5.68    3.35  20.19     3       3     3
vgg16        36.46   31.75  75.61  0.47    2.03   2.68     3       3     3

inflation from leakage (macro F1 points):
             slice - patient  exam - patient
arch                                        
inceptionv3            23.27           11.28
lungcnet               32.24            5.17
mobilenetv2            18.62            5.57
vgg16                  43.85            4.71

collapsed runs (report these, do not silently average them):
       arch     arm  seed  val_accu

In [ ]:
!python src/analyse.py --results "/content/drive/MyDrive/leakage_bench/results_nlst.csv" --out tables/

macro F1 by architecture and split arm

       arch     arm  macro_f1_mean  macro_f1_sd  accuracy_mean  above_baseline_mean  epochs_mean  n_seeds
inceptionv3   slice          56.56         0.31          59.41                19.65        12.33        3
inceptionv3    exam          44.57         2.81          49.54                 9.89         7.33        3
inceptionv3 patient          33.29         3.49          42.49                 2.73         6.33        3
   lungcnet   slice          62.56         0.98          64.60                24.84        20.67        3
   lungcnet    exam          35.49         1.19          48.05                 8.39         6.67        3
   lungcnet patient          30.32         1.87          41.37                 1.60         6.33        3
mobilenetv2   slice          54.09        20.20          57.97                18.22        27.33        3
mobilenetv2    exam          41.04         5.68          50.11                10.46         6.00        3
mobile

In [10]:
import os
print(os.path.exists('/content/lungcnet-leakage/src/analyse.py'))
print(os.path.exists('/content/drive/MyDrive'))

True
True


In [11]:
!cd /content/lungcnet-leakage && python src/analyse.py --results "/content/drive/MyDrive/leakage_bench/results_nlst.csv" --out "/content/drive/MyDrive/leakage_bench/tables"

macro F1 by architecture and split arm

       arch     arm  macro_f1_mean  macro_f1_sd  accuracy_mean  above_baseline_mean  epochs_mean  n_seeds
inceptionv3   slice          56.56         0.31          59.41                19.65        12.33        3
inceptionv3    exam          44.57         2.81          49.54                 9.89         7.33        3
inceptionv3 patient          33.29         3.49          42.49                 2.73         6.33        3
   lungcnet   slice          62.56         0.98          64.60                24.84        20.67        3
   lungcnet    exam          35.49         1.19          48.05                 8.39         6.67        3
   lungcnet patient          30.32         1.87          41.37                 1.60         6.33        3
mobilenetv2   slice          54.09        20.20          57.97                18.22        27.33        3
mobilenetv2    exam          41.04         5.68          50.11                10.46         6.00        3
mobile

## 5. Control: 1-NN on raw pixels

Separates memorisation from learning. A 1-nearest-neighbour classifier on
32x32 greyscale pixels cannot learn a diagnostic feature; it can only retrieve
the most similar image it has already seen. Run through the same three arms on
the same seeds, it shows whether the slice-arm advantage is retrieval of
near-duplicate slices. Also reports how often each test image's nearest
neighbour came from its own patient and its own exam.

CPU only. Redraws the splits deterministically if `data/splits_nlst.csv` is absent.


In [ ]:
# src/control_knn.py must be committed to the repo first
!cd /content/lungcnet-leakage && git pull -q

import os
assert os.path.exists('/content/lungcnet-leakage/src/control_knn.py'), (
    'src/control_knn.py is not in the repo yet -- upload it on GitHub, then re-run this cell'
)

!cd /content/lungcnet-leakage && python src/control_knn.py \
    --config configs/nlst.yaml \
    --out "/content/drive/MyDrive/leakage_bench/results_control_knn.csv"
